# Organelle Contact Analysis 🔗
-------

## **Objectives**
In this notebook, the logic for identifying and quantifying *organelle contact sites* -- regions of overlap -- between 2 or more organelles is outlined. The notebook begins with an explanation of how to produce 2-way contact sites, expands upon that to produce multi-way contacts, and then outlines the methods to quantify the amount (count, volume), morphology (size and shape), and subcellular distribution.

### **Summary of quantification steps:**
1. *Imports:* import packages
2. *Inputs:* Segmentation images of organelles invovled in contacts
    - Load intensity and segmentation images
    - Visualize inputs
    - Format images and names into a dictionary
    - Create contact site objects
3. *Run Analysis:* Quantify amount, size, shape, distribution
   - Run skimage.measure.regionprops to quantify amount, size, and shape (see Quantification Notebook ____ for more details)
   - Run XY and Z distribution analysis (see Quantification Notebook ____ for more details)
4. *Define Function:* get_contact_metrics_3D()
5. *Outputs:* Dataframe of organelle contact amount, size, shape, and distribution

## **Imports**
The following packages are necessary for contact site analysis.

In [ ]:
import warnings
import numpy as np
from typing import Any, List, Union, Dict
import pandas as pd
from IPython.display import display
from pathlib import Path
import os, sys
import itertools 
import time

import napari
from napari.utils.notebook_display import nbscreenshot

from skimage.measure import regionprops_table, label
from skimage.segmentation import watershed

from infer_subc.core.file_io import (read_czi_image,
                                     import_inferred_organelle,
                                     list_image_files)

from infer_subc.core.img import *
from infer_subc.utils.stats import *
from infer_subc.utils.stats import (_assert_uint16_labels)
from infer_subc.utils.stats_helpers import *

from infer_subc.organelles import * 

# from infer_subc.core.file_io import read_czi_image, read_tiff_image
# from infer_subc.core.img import apply_mask
# from infer_subc.utils.batch import list_image_files, find_segmentation_tiff_files
# from infer_subc.utils.stats import surface_area_from_props, get_XY_distribution, get_Z_distribution, _assert_uint16_labels

#For Convexhull Errors
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

viewer = napari.Viewer()

pd.options.display.max_columns = None

### TODO: need to move this into the code
def _import_inferred_organelle(name: str, suffix: str, meta_dict: Dict, out_data_path: Path, file_type: str) -> Union[np.ndarray, None]:
    """
    read inferred organelle from ome.tif file

    Parameters
    ------------
    name: str
        name of organelle.  i.e. nuc, lyso, etc.
    suffix: str
        the splitter between the file name and segmentation suffix (e.g., "-" if the segmentation name was "01_condition1-mito")
    meta_dict:
        dictionary of meta-data (ome) from original file
    out_data_path:
        Path object of directory where tiffs are read from
    file_type: 
        The type of file you want to import as a string (ex - ".tif", ".tiff", ".czi", etc.)

    Returns
    -------------
    exported file name

    """

    # copy the original file name to meta
    img_name = Path(meta_dict["file_name"])  #
    # add params to metadata
    if name is None:
        pass
    else:
        organelle_fname = f"{img_name.stem}{suffix}{name}{file_type}"

        organelle_path = out_data_path / organelle_fname

        if Path.exists(organelle_path):
            # organelle_obj, _meta_dict = read_ome_image(organelle_path)
            organelle_obj = read_tiff_image(organelle_path)  # .squeeze()
            print(f"loaded  inferred {len(organelle_obj.shape)}D `{name}`  from {out_data_path} ")
            return organelle_obj
        else:
            print(f"`{name}` object not found: {organelle_path}")
            raise FileNotFoundError(f"`{name}` object not found: {organelle_path}")

## **Inputs:**
To determine which organelles are contacting each other, or are in close enough proximity to make contacts possible, segmentations of the organelles are required. The segmentations define the edges of the organelles which can then be used to determine proximity between other organelle objects. Here, we consider a contact site to be an overlap between organelle objects.

### **Load intensity images**

In [ ]:
# Define the path to your image data; this directory location should contain subfolders for the "raw" intensity data, the segmenation images, and a folder to save the quantification output
data_root_path = Path(os.path.expanduser("~")) / "Documents/Python Scripts/Infer-subc-2D"

# Specify the raw file path and list all files of a specified type
raw_path = data_root_path / "raw_single"
im_type = ".tiff"
img_file_list = list_image_files(raw_path, im_type)

# Specify the segmentation file path
seg_path = data_root_path/"out_single"
seg_suffix = "-"

# Specify the output file path
out_data_path = data_root_path / "quant_single"
if not Path.exists(out_data_path):
    Path.mkdir(out_data_path)
    print(f"making {out_data_path}")

In [ ]:
# Specify the index of the raw image you would like to use for testing & import it
test_img_n = 0
test_img_name = img_file_list[test_img_n]
img_data,meta_dict = read_czi_image(test_img_name)

# Some metadata
channel_names = meta_dict['name']
img = meta_dict['metadata']['aicsimage']
scale = meta_dict['scale']
channel_axis = meta_dict['channel_axis']

### **Load segmentation images**

In [ ]:
# List which organelles will be included in the contact analysis by including the segmentation image suffix here
org_names = ['LD', 'ER', 'golgi', 'lyso', 'mito', 'perox']
seg_file_type = ".tiff"

# List cell regions included in the analysis, including a masking object (cell or cytoplasm) and a centering object for XY distribution measurements
region_names = ['nuc', 'cell']
mask = 'cell'
dist_centering_obj = 'nuc'

# Import segmentation images associated to the test image selected above
org_segs = [_import_inferred_organelle(org, seg_suffix, meta_dict, seg_path, seg_file_type) for org in org_names]
region_segs = [_import_inferred_organelle(reg, seg_suffix, meta_dict, seg_path, seg_file_type) for reg in region_names]
mask = region_segs[region_names.index(mask)]
center_obj = region_segs[region_names.index(dist_centering_obj)]

In [ ]:
for num, org in enumerate(org_segs):
    viewer.add_labels(org_segs[num], name=org_names[num])

for num, org in enumerate(region_segs):
    viewer.add_labels(region_segs[num], name=region_names[num])

### **Make dictionaries to store images and names**
Here we will convert the input list of __organelle segmentations__ and the input list of __organelle names__ into a dictionary where the key is the segmentation suffix and the value is the segmentation image as a np.array.

In [ ]:
organelle_segs = {}                                                       #Initialize dictionary
for idx, name in enumerate(org_names):                                  #Loop across each organelle name
    if name == 'ER':                                                    #Proceed only for ER
        organelle_segs[name]=(org_segs[idx]>0).astype(np.uint16)          #Ensures ER is labeled only as one object & sets it as key for its object segmentation
    else:                                                               #Proceed for other organelles
        organelle_segs[name]=org_segs[idx]                                #Set the organelle name as the key for the corresponding object segmentation

In [ ]:
def _make_contact_dict(obj_names: list[str],                                        #Intakes list of object names
                       obj_segs: list[np.ndarray]):                                 #Intakes list of object segmentations
    objs_labeled = {}                                                       #Initialize dictionary
    for idx, name in enumerate(obj_names):                                  #Loop across each organelle name
        if name == 'ER':                                                    #Proceed only for ER
            objs_labeled[name]=(obj_segs[idx]>0).astype(np.uint16)          #Ensures ER is labeled only as one object & sets it as key for its object segmentation
        else:                                                               #Proceed for other organelles
            objs_labeled[name]=obj_segs[idx]                                #Set the organelle name as the key for the corresponding object segmentation
    return objs_labeled                                                 #Return a dictionary of segmented objects with keys as the organelle name

In [ ]:
labeled_dict = _make_contact_dict(org_names, org_segs)

### **Visualize input images**

In [ ]:
viewer.layers.clear()
viewer.add_image(img_data)

for m, reg in zip(region_names, region_segs):
    viewer.add_labels(reg, name=m)

for n in org_names:
    img = labeled_dict[n]
    viewer.add_labels(img, name=n)

# nbscreenshot(viewer, canvas_only=True)

### **Create contact sites**
All possible nonredundant combinations of organelles will be created from on a list of organelle suffix strings. The combinations will begin with all possible 2-way combinations and go up to the nth-order combination where n is the number of organelles in the original list.

> ***Naming scheme for contact sites:***
>
> The format for the contact site string is as follows:
> <br>
> `organelle1` + `splitter` + `organelle2`
> 
> For example, if the desired contact site is between the _endoplasmic reticulum (ER)_ and _peroxisomes ('perox' or 'PO')_, and in the above block of code the _endoplasmic reticulum_ is set equal to _`ER`_, the _splitter_ is listed as _`_`_, and _peroxisomes_ are set equal to _`perox`_ in the above block of code, the contact site string would be as follows:
> <br>
> `ER_perox`

Before creating the list of possible contacts, you have to decide what the splitter is going to be. 

Enter the character you'd like to have as the splitter in the below block of code:

In [ ]:
# Specify the character to use as the separation between organelles in the contact site names
splitter = "_"

Now, we can run through the code that creates the list of possible contacts.

In [ ]:
# Find all possible contact combinations to produce a list of all possible contact site names
all_pos =[]
for n in list(map(lambda x:x+2, (range(len(org_names)-1)))):
        all_pos += itertools.combinations(org_names, n)

possib = [splitter.join(cont) for cont in all_pos]
print(list(enumerate(possib)))

Notice above the numbers associated with each contact pairing? 

Below, set the desired contact pairing's number equal to the num variable:

In [ ]:
num = 12

#### **EXAMPLE: A 2-way Contact Site**
**Create the contact sites:** 
<br>First, we will run through a simple example of how to idenitfy a single 2-way contact site.

In [ ]:
# Specify one contact site to examine in the following 2-way analysis and the splitter used
orgs = possib[num]

# Create a contact site between the selected organelles
org_A = organelle_segs[orgs.split(splitter)[0]]
org_B = organelle_segs[orgs.split(splitter)[1]]
site = org_A * org_B

In [ ]:
viewer.layers.clear()
viewer.add_image(org_A>0, colormap='green', blending='additive')
viewer.add_image(org_B>0, colormap='magenta', blending='additive')
viewer.add_image(site>0, blending='additive')

# nbscreenshot(viewer, canvas_only=True)

**If there are higher order contacts to assess:** 
<br>If there are more than two organelles included in the contacts analysis, we need to determine if the 2-way sites are also involved in higher order contacts (e.g., 3-way, 4-way, etc.). 

We begin by creating the 3-way sites from the 2-way sites. This includes first iterating across the organelles in our original list of organelle names and, if the name is not present in the original 2-way contact name, creating the overlap between the 2-way site and the third organelle. The 2-way sites that include a third organelle are then removed from the 2-way contact site image using a watershed function. The sites that are remaining represent true 2-way sites that are not involved in a higher order contact. They will be listed in the final dataframe as non-redundant.

In [ ]:
# list all lower order contacts
LOc_NR = site.copy()

# Loop through the list of organelles to create 3way contacts
for org, val in organelle_segs.items():
    if (org not in orgs.split(splitter)) and np.any(site*val):

        # Create the 3-way contact site
        HOc = site*val

        # Use watershed to select all 2-way sites that include a third organelle
        HOc_expanded = watershed(image=np.invert(site),
                                 markers=HOc,
                                 mask=site,
                                 connectivity=np.ones((3, 3, 3), bool))>0
        
        # Remove those sites from the original 2-way sites & repeat to produce the lower order non-redundant contacts only
        LOc_NR = LOc_NR * (np.invert(HOc_expanded))

Now, we can visualize the 2-Way contact sites and the non-redundant 2-way contact sites with the code below:

In [ ]:
viewer.layers.clear()
viewer.add_image(site>0, blending='additive', colormap='green', name='All 2-way Sites')
viewer.add_image(LOc_NR>0, blending='additive', colormap='magenta', name='Non-redundant 2-way Sites')

# nbscreenshot(viewer, canvas_only=True)

#### **EXAMPLE: A 3-way Contact Site**
**Create the contact sites:** 
<br>The above logic can be applied to higher order contacts as well.

Run the below line of code to check for possible contacts between your organelles and the number associated to them:

In [ ]:
print(enumerate(possib))

Just like before, set the num variable equal to the desired contact type's number, but this time look exclusively at the contacts that are higher order (for this example, lets choose a grouping of 3 organelles):

In [ ]:
num = 25

The below code will then apply the same logic used for 2 way contacts to the n-way contacts.

In [ ]:
# Specifying one 3-way contact site name
threeway_names = possib[num]

# Creating the three way contact site objects
three_site = np.ones_like(labeled_dict[threeway_names.split(splitter)[0]])
for org in threeway_names.split(splitter):
    # Creating desired overlap regions
    b = organelle_segs[org]         #collects organelle b
    c = (site>0)*(b>0)              #determines true overlapping area without labels
    digit = len(str(np.max(site)))  #finds number of digits in largest valued label of organelle a segmentation
    site = (b*(10**(digit+1)))+site #assigns unique labels to each overlap
    site[c.astype(bool)==False] = 0 #ensures that locations with no overlap are labeled as 0
    site = label(site)              #simplifies labels

# Selecting only the non-redundant 3-way contacts for reference in the dataframe later
threeway_NR = three_site.copy()
for org, val in labeled_dict.items():
    if (org not in threeway_names.split(splitter)) and np.any(three_site*val):
        threeway_NR = threeway_NR * (np.invert(watershed(image=np.invert(three_site),
                                                markers=(three_site*val),
                                                mask=three_site,
                                                connectivity=np.ones((3, 3, 3), bool))>0))

Now, we can visualize the 3 way sites and the non-redundant 2-way sites with the below code:

In [ ]:
viewer.layers.clear()
viewer.add_image(three_site>0, blending='additive', colormap='green', name='All 2-way Sites')
viewer.add_image(threeway_NR>0, blending='additive', colormap='magenta', name='Non-redundant 2-way Sites')

# nbscreenshot(viewer, canvas_only=True)

Using the above logic, we can now create a function that will perform the steps for us. This function is used as the basis for batch processing and for analyzing the contacts in infer-subc.

In [ ]:
def _create_contact(orgs:str,
                    organelle_segs: dict[str:np.ndarray],
                    splitter: str="X") -> tuple[np.ndarray, np.ndarray]: 
    ##########################################
    ## CREATE CONTACT
    ##########################################
    site = np.ones_like(organelle_segs[orgs.split(splitter)[0]])
    for org in orgs.split(splitter):
        # Creating desired overlap regions
        b = organelle_segs[org]             #collects organelle b
        valid = (b>0)*(site>0)
        digit = len(str(np.max(site)))      #finds number of digits in largest valued label of organelle a segmentation
        site = (b*(10**(digit)))+site       #assigns unique labels to each overlap
        site[valid.astype(bool)==False]=0   #ensures that locations with no overlap are labeled as 0
        site = label(site)                  #simplifies labels
    ##########################################
    ## DETERMINE REDUNDANT CONTACTS
    ##########################################
    LOc_NR = site.copy()
    for org, val in organelle_segs.items():
        if (org not in orgs.split(splitter)) and np.any(site*val):
            print(f"Examining {org} Higher Order contacts", end="\r")
            digit = len(str(np.max(val)))
            valid = (LOc_NR>0)*(val>0)
            HOc = (LOc_NR*(10**(digit)))+val
            HOc[valid.astype(bool)==False]=0
            HOc = label(HOc)
            maxi = len(np.unique(LOc_NR[HOc>0]))
            for num, id in enumerate(np.unique(LOc_NR[HOc > 0])):
                per = round((100*((num+1)/maxi)), 2)
                inver = np.ones_like(LOc_NR)
                inver[LOc_NR==id] = 0
                LOc_NR = LOc_NR*inver
                print(f"Examining {org} Higher Order contacts: {per}% complete", end="\r")
            print(f"Examining {org} Higher Order contacts: {100.00}% complete")
    return site, LOc_NR

Next, lets check if the function works the same as the code it was based off of:

In [ ]:
threesite_test, threesite_NR = _create_contact(possib[num], labeled_dict)

print(f"Are the function's contact export and the code its based off of are equal in value? {np.array_equal(threesite_test, three_site)}")
print(f"Are the function's nonredundant export and the code its based off of equal in value? {np.array_equal(threesite_NR, threeway_NR)}")

## **Run the analysis:**
Now that we have a method to create contact sites -- both redundant and not --, we can quantify the amount, size, shape, and distribution of them using methods from the following Notebooks:
- Amount, size, and shape: _________
- Distribution: _____________

### **Quantify contact site *amount*, *size*, and *shape*:**
The analysis included here will utilize the 3-way contacts created about: `threesite_test` and `threesite_NR`

In [ ]:
# We first need to apply a mask to confine our analysis to our one cell of interest
labels = label(apply_mask(threesite_test, mask)).astype("int")
para_labels = apply_mask((threesite_NR>0), mask).astype("int") * labels

In [ ]:
# Run skimage.regionprops to quantify the amount, size, and shape of the contact site objects
properties = ["label", 
              "centroid", "bbox",
              "area", "equivalent_diameter",
              "extent", "euler_number", "solidity", "axis_major_length", "slice"]

props = regionprops_table(labels, intensity_image=None, properties=properties, extra_properties=None, spacing=scale)

surface_area_tab = pd.DataFrame(surface_area_from_props(labels, props, scale))

#### **List which organelles are involved in each contact site:**
Because there are many organelles per image each labeled with a unique ID number, it is helpful to understand which of those are involved in each particular contact site. The organelle ID numbers will be encoded as a unique organelle ID as follows: if peroxisome ID#6 was involved in a contact with the ER (only one object) and mitochondria ID#43, the contact would be listed as "ER_perox_mito" and the label for the contact would be listed as 1_6_43. 

This section of code will also creates the column of data about whether a contact site is redundant or not. As a reminder, any contact that is also involved in a higher order contact will be labled 'True' for redundant. This does not remove redundant contacts from analysis--it only provides additional context to the data.

In [ ]:
cont_inv = []                                                   # initializes a variable to be used for creating the values for the contacts in the dictionary
                                                                # This variable is a list of the labels of each organelle involved in one contact site between those organelles
involved = threeway_names.split(splitter)                       # creates list of all involved organelles in the contact
indexes = dict.fromkeys(involved, [])                           # A dictionary of indexes of site involved in the contact
indexes[threeway_names] = []                                    #   str = "contact" or "organelle", may have multiple different organelles 
                                                                #   list = contact or organelle number in image corresponding to the same contact in the other keys
                                                                #   a 2-way contact will have 3 keys, a 3-way contact will have 4 keys, etc
redundancy = []
for index, l in enumerate(props["label"]):
    cont_inv.clear()                                            #clears cont_inv variable of any labels from past contact site for new contact site
    present = (para_labels[props["slice"][index]])              #examines contact site to find if it is present in a higher order contact or not
    present = present==l
    redundant = not np.any(present)
    redundancy.append(redundant)
    for org in involved:                                        #iterates across list of involved organelles
        volume = (labels[props["slice"][index]])
        lorg = labeled_dict[org][props["slice"][index]]
        volume = volume==l
        lorg = lorg[volume]
        all_inv = np.unique(lorg[lorg>0]).tolist()
        if len(all_inv) != 1:                                   #ensures only one label is involved in the contact site
            print(f"we have an error.  as-> {all_inv}")         #informs the console of any errors and the reasoing for it
        indexes[org].append(all_inv[0])                         #adds the label of the organelle involved in the contact to the organelle's key's list
        cont_inv.append(f"{all_inv[0]}")                        #adds the label of the organelle involved in the contact to the list of involved organelle labels
    indexes[threeway_names].append("_".join(cont_inv))          #adds the combination of all the organelle's labels involved in the contact to the contact key's list

##### TODO: figure out why this isn't correct; honestly just remove it if it isn't necessary
indexes['lyso'] == indexes['ER'] == indexes['golgi']

#### **Configure the dataframe with the above information**
Here, all of the individual datatables created to examine the contacts data present are combined into one data table for export.

In [ ]:
props_table = pd.DataFrame(props)
props_table.drop(columns=['slice', 'label'], inplace=True)
props_table.insert(0, 'label',value=indexes[threeway_names])
props_table.insert(0, "object", threeway_names)
props_table.rename(columns={"area": "volume"}, inplace=True)
props_table.insert(11, "surface_area", surface_area_tab)
props_table.insert(13, "SA_to_volume_ratio", props_table["surface_area"].div(props_table["volume"]))
if scale is not None:
    round_scale = (round(scale[0], 4), round(scale[1], 4), round(scale[2], 4))
    props_table.insert(loc=2, column="scale", value=f"{round_scale}")
else: 
    props_table.insert(loc=2, column="scale", value=f"{tuple(np.ones(labels.ndim))}") 
props_table.insert(2, "in_higher_order", list(map(bool, redundancy)))

props_table.head()

### **Quantify Distribution of Contact Sites**
The relative subcellular localization of each contact site will be quantified with respect to XY and Z using the distribution measurements outlines in ___________ notebook. This analysis is optional for contact sites in the final function defined below.

In [ ]:
# Specify the distribution settings here:
include_contact_dist = True
dist_centering_obj = 'nuc'
dist_center_on = False
dist_keep_center_as_bin = True
dist_num_bins = 5
dist_zernike_degrees=None

# Run the distribution analysis here:
dist_tabs = []
if include_contact_dist:
    XY_contact_dist, XY_bins, XY_wedges = get_XY_distribution(mask=mask, 
                                                              obj=threesite_test,
                                                              obj_name=threeway_names,
                                                              centering_obj=center_obj,
                                                              scale=scale,
                                                              center_on=dist_center_on,
                                                              keep_center_as_bin=dist_keep_center_as_bin,
                                                              num_bins=dist_num_bins,
                                                              zernike_degrees=dist_zernike_degrees)
          
    Z_contact_dist = get_Z_distribution(mask=mask,
                                        obj=threesite_test,
                                        obj_name=threeway_names,
                                        center_obj=center_obj,
                                        scale=scale)
    contact_dist_tab = pd.merge(XY_contact_dist, Z_contact_dist, on=["object", "scale"])
    dist_tabs.append(contact_dist_tab)

combined_dist_tab = pd.concat(dist_tabs, ignore_index=True)
combined_dist_tab.insert(loc=0,column='image_name',value=test_img_name.stem)

combined_dist_tab

## **Define the Function:**
This section creates new functions that combines the above steps for easy processing of contacts. The function is then included in the code and tested.

In [ ]:
def _get_contact_metrics_3D(list_obj_names: List[str],
                            list_obj_segs: List[np.ndarray],
                            list_region_names: List[str],
                            list_region_segs: List[np.ndarray],
                            mask: np.ndarray,
                            img_f: str,
                            splitter: str="X",
                            scale: Union[tuple, None]=None,
                            include_dist:bool=False, 
                            dist_centering_obj: Union[np.ndarray, None]=None,
                            dist_num_bins: Union[int, None]=None,
                            dist_zernike_degrees: Union[int, None]=None,
                            dist_center_on: Union[bool, None]=None,
                            dist_keep_center_as_bin: Union[bool, None]=None) -> list:
    
    """ 
    Parameters:
    ----------
    list_obj_names: List[str],
        A list of object names to include in the contact analysis
    list_obj_segs: List[np.ndarray],
        A list of numpy arrays associated to each of the names in list_obj_names. The expectation is that these images will be binary segmentation or uint16 labeled images.
    list_region_names: List[str],
        A list of regions to analysis from. Current versions will only utilize:
            - the mask object (identifying the area to analyze from) and
            - a centering object (used to define the center of the XY region for distribution analysis; if none is defined, the default will be the center of the mask region)
    list_region_segs: List[np.ndarray],
        A list of numpy arrays associated to each of the region names. The expectations is that these images will be a binary image. The masking object is expected to be only one object per image.
    mask: np.ndarray,
        The name of the object to use for masking
    splitter: str="_",
        The character you would like to separate object names in the new contact sites names.
        Ex) splitter="_"
            list_obj_names=['ER', 'golgi']
            --> 'ER_golgi'
    scale: Union[tuple, None]=None,
        The dimentions of the image voxels (Z,Y,X). If not scale, the default is (1,1,1)
    include_dist:bool=False, 
        True --> quantify the distribution of contact sites within the masked region
        False --> do not quantify distributions
    dist_centering_obj: Union[np.ndarray, None]=None,
        See get_XY_distribution() for information.
    dist_num_bins: Union[int, None]=None,
        See get_XY_distribution() for information.
    dist_zernike_degrees: Union[int, None]=None,
        See get_XY_distribution() for information.
    dist_center_on: Union[bool, None]=None,
        See get_XY_distribution() for information.
    dist_keep_center_as_bin:
        See get_XY_distribution() for information.

    Output:
    -------
    --> props_table: pd.Dataframe()
        A Pandas dataframe containing quantification of the amount, size, and shape of each contact site.
    dist_tabs: pd.Dataframe() (ONLY IF include_dist=True)
        A Pandas dataframe containing quantification of the contact sites distribution per cell in XY and Z.
    """

    # Properties to measure in regionprops
    properties = ["label", "centroid", "bbox", "area", 
                "equivalent_diameter", "extent", "euler_number", 
                "solidity", "axis_major_length", "slice"]
    
    # preparing mask object
    mask = list_region_segs[list_region_names.index(mask)]

    # preparing centering object
    center_obj = list_region_segs[list_region_names.index(dist_centering_obj)]


    ##########################################
    ## CREATING CONTACT SITE LIST
    ##########################################
    org_dict = _make_contact_dict(list_obj_names, list_obj_segs)

    all_pos =[]
    for n in list(map(lambda x:x+2, (range(len(list_obj_names)-1)))):
        all_pos += itertools.combinations(list_obj_names, n)
    possib = [splitter.join(cont) for cont in all_pos]


    ##########################################
    ## LOOP THROUGH EACH SITE AND QUANTIFY
    ##########################################
    props_tabs = []
    dist_tabs = []
    for cont in possib:
        print(cont)
        site, LOc_NR = _create_contact(cont, org_dict, splitter)
        labels = label(apply_mask(site, mask)).astype(int)
        para_labels = apply_mask((LOc_NR>0), mask).astype(int) * labels
    

        ## quantify amount, size, and shape
        props = regionprops_table(labels, intensity_image=None, properties=properties, extra_properties=None, spacing=scale)
        surface_area_tab = pd.DataFrame(surface_area_from_props(labels, props, scale))


        ## LIST WHICH ORGANELLES ARE INVOLVED IN THE CONTACT
        cont_inv = []
        involved = cont.split(splitter)
        indexes = dict.fromkeys(involved, [])
        indexes[cont] = []

        redundancy = []
        for index, l in enumerate(props["label"]):
            cont_inv.clear()
            present = para_labels[props["slice"][index]]
            present = present==l
            redundant = not np.any(present)
            redundancy.append(redundant)
            for org in involved:
                volume = labels[props["slice"][index]]
                lorg = org_dict[org][props["slice"][index]]
                volume = volume==l
                lorg = lorg[volume]                                 
                all_inv = np.unique(lorg[lorg>0]).tolist()          
                if len(all_inv) != 1:
                    print(f"we have an error.  as-> {all_inv}")
                indexes[org].append(all_inv[0])
                cont_inv.append(f"{all_inv[0]}")
            indexes[cont].append(splitter.join(cont_inv))
        

        ## CREATE COMBINED DATAFRAME OF THE QUANTIFICATION
        props_table = pd.DataFrame(props)
        props_table.drop(columns=['slice', 'label'], inplace=True)
        props_table.insert(0, 'label',value=indexes[cont])
        props_table.insert(0, "object", cont)
        props_table.rename(columns={"area": "volume"}, inplace=True)
        props_table.insert(11, "surface_area", surface_area_tab)
        props_table.insert(13, "SA_to_volume_ratio", 
        props_table["surface_area"].div(props_table["volume"]))
        if scale is not None:
            round_scale = (round(scale[0], 4), round(scale[1], 4), round(scale[2], 4))
            props_table.insert(loc=2, column="scale", value=f"{round_scale}")
        else: 
            props_table.insert(loc=2, column="scale", value=f"{tuple(np.ones(labels.ndim))}")
        props_table.insert(2, "in_higher_order", list(map(bool, redundancy)))
        props_tabs.append(props_table)


        ## optional: DISTRIBUTION OF CONTACTS MEASUREMENTS
        if include_dist:
            XY_contact_dist, XY_bins, XY_wedges = get_XY_distribution(mask=mask, 
                                                                      obj=site,
                                                                      obj_name=cont,
                                                                      centering_obj=center_obj,
                                                                      scale=scale,
                                                                      center_on=dist_center_on,
                                                                      keep_center_as_bin=dist_keep_center_as_bin,
                                                                      num_bins=dist_num_bins,
                                                                      zernike_degrees=dist_zernike_degrees)
        
            Z_contact_dist = get_Z_distribution(mask=mask,
                                                obj=site,
                                                obj_name=cont,
                                                center_obj=center_obj,
                                                scale=scale)
            contact_dist_tab = pd.merge(XY_contact_dist, Z_contact_dist, on=["object", "scale"])
            dist_tabs.append(contact_dist_tab)

        indexes.clear()
    

    ## JOIN PER SITE TABLES TO CREATE TWO FINAL TABLES
    props_tables = pd.concat(props_tabs)
    props_tables.insert(loc=0,column='image_name',value=img_f.stem)


    if include_dist:
        dist_tables = pd.concat(dist_tabs)
        dist_tables.insert(loc=0,column='image_name',value=img_f.stem)
        return props_tables, dist_tables
    else:
        return props_tables

In [ ]:
cont_tab, dist_tab = _get_contact_metrics_3D(org_names,
                                             org_segs,
                                             region_names,
                                             region_segs,
                                             'cell',
                                             "X",
                                             scale,
                                             True,
                                             'nuc',
                                             5,
                                             None,
                                             False,
                                             True)

## **OUTPUT:**
The tables below show the expected output from this notebook. The analysis was carried out on one image; batch processing is possible and outlined in notebook _____. 

DEFINITIONS:
`TODO: we need to write out the definitions of all the columns in the table here`


In [ ]:
cont_tab.head()

In [ ]:
dist_tab.head()

## **Batch Processing:**

In [ ]:
def ncont_batch(out_file_name: str,
                raw_path: str,
                seg_path: str,
                out_path: str,
                mask: str,
                organelle_names: list[str],
                masks_file_name: list[str],
                raw_file_type: str=".tiff",
                scale: bool=True,
                splitter: str="X",
                include_contact_dist: bool=True,
                centering: Union[str, None]=None,
                num_bins: Union[int, None]=None,
                zernike_degrees: Union[int, None]=None,
                center_on: Union[bool, None]= None,
                center_as_bin: Union[bool, None]= None,
                seg_suffix: Union[str, None]="-"):
    start = time.time()
    count = 0

    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(seg_path, str): seg_path = Path(seg_path)
    if isinstance(out_path, str): out_path = Path(out_path)

    if not Path.exists(out_path):
        Path.mkdir(out_path)
        print(f"making {out_path}")
    
    # reading list of files from the raw path
    img_file_list = list_image_files(raw_path, raw_file_type)

    # list of segmentation files to collect
    segs_to_collect = organelle_names + masks_file_name

    contact_tabs = []
    distrib_tabs = []
    for img_f in img_file_list:
        print(img_f.stem)
        count = count + 1
        filez = find_segmentation_tiff_files(img_f, segs_to_collect, seg_path, seg_suffix)

        # read in raw file and metadata
        img_data, meta_dict = read_czi_image(filez["raw"])

        # define the scale
        if scale:
            scale_tup = meta_dict['scale']
        else:
            scale_tup = None
        
        regions = [read_tiff_image(filez[region]) for region in masks_file_name]
        organelles = [read_tiff_image(filez[org]) for org in organelle_names]

        ####################################################################################################
        if include_contact_dist:
            for org in (organelle_names):
                XY_org_distribution, XY_bin_masks, XY_wedge_masks = get_XY_distribution(mask=regions[masks_file_name.index(mask)],
                                                                                    centering_obj=regions[masks_file_name.index(centering)],
                                                                                    obj=organelles[organelle_names.index(org)],
                                                                                    obj_name=org,
                                                                                    scale=scale_tup,
                                                                                    num_bins=num_bins,
                                                                                    center_on=center_on,
                                                                                    keep_center_as_bin=center_as_bin,
                                                                                    zernike_degrees=zernike_degrees)
                Z_org_distribution = get_Z_distribution(mask=regions[masks_file_name.index(mask)], 
                                                        obj=organelles[organelle_names.index(org)],
                                                        obj_name=org,
                                                        center_obj=regions[masks_file_name.index(centering)],
                                                        scale=scale_tup)
            
                org_distribution_metrics = pd.merge(XY_org_distribution, Z_org_distribution,on=["object", "scale"])

                distrib_tabs.append(org_distribution_metrics)
        
        #####################################################################################################
        if len(organelle_names) >= 2:
            if include_contact_dist:
                cont_tabs, dist_tabs = _get_contact_metrics_3D(list_obj_names = organelle_names,
                                                               list_obj_segs = organelles,
                                                               list_region_names = masks_file_name,
                                                               list_region_segs= regions,
                                                               mask = mask,
                                                               img_f = img_f,
                                                               splitter = splitter,
                                                               scale = scale_tup,
                                                               include_dist = include_contact_dist, 
                                                               dist_centering_obj = centering,
                                                               dist_num_bins = num_bins,
                                                               dist_zernike_degrees = zernike_degrees,
                                                               dist_center_on = center_on,
                                                               dist_keep_center_as_bin = center_as_bin)
            else:
                cont_tabs = _get_contact_metrics_3D(list_obj_names = organelle_names,
                                                    list_obj_segs = organelles,
                                                    list_region_names = masks_file_name,
                                                    list_region_segs= regions,
                                                    mask = mask,
                                                    img_f = img_f,
                                                    splitter = splitter,
                                                    scale = scale_tup,
                                                    include_dist = include_contact_dist, 
                                                    dist_centering_obj = centering,
                                                    dist_num_bins = num_bins,
                                                    dist_zernike_degrees = zernike_degrees,
                                                    dist_center_on = center_on,
                                                    dist_keep_center_as_bin = center_as_bin)
        #####################################################################################################
        contact_tabs.append(cont_tabs)
        cont_tabs.head()
        if include_contact_dist:
           distrib_tabs.append(dist_tabs)
        end2 = time.time()
        print(f"Completed processing for {count} images in {(end2-start)/60} mins.")


    final_contact = pd.concat(contact_tabs, ignore_index=True)
    contact_csv_path = out_path / f"{out_file_name}contacts.csv"
    final_contact.to_csv(contact_csv_path)

    if include_contact_dist:
        final_dist = pd.concat(distrib_tabs, ignore_index=True)
        dist_csv_path = out_path / f"{out_file_name}distributions.csv"
        final_dist.to_csv(dist_csv_path)
    
    end = time.time()
    print(f"Quantification for {count} files is COMPLETE! Files saved to '{out_path}'.")
    print(f"It took {(end - start)/60} minutes to quantify these files.")
    return cont_tabs

In [6]:
ncont_batch(out_file_name ="declumpeddata101824",
            raw_path ="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/raw_single",
            seg_path ="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/out_single",
            out_path ="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/quant_single",
            mask = "cell",
            organelle_names = ["ER","golgi", "LD", "lyso", "mito", "nuc", "perox"],
            masks_file_name = ["cell","nuc"],
            raw_file_type = ".tiff",
            scale = True,
            splitter = "X",
            include_contact_dist = False,
            centering = "nuc",
            num_bins = 5,
            zernike_degrees = None,
            center_on = False,
            center_as_bin = True).head()

KeyboardInterrupt: 